# EXP-014 dev-small 확인: RRF k 60 → 20

EXP-011(k 60)과 EXP-014(k 20)의 dev-small 지표와 질의별 상위 순위 이동을 본다. 판정에 쓰지 않고, k는 여기서 크게 무너질 때만 바꾼다(`docs/experiments/EXP-014.md`).

In [1]:
from pathlib import Path

import pandas as pd

from rag.eval.metrics import gold_ranks, load_cfg, load_gold, read_run

In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
PREFIX = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256'
RUN_FILES = {'EXP-011 (k 60)': f'{PREFIX}-dev-small.csv',
             'EXP-014 (k 20)': f'{PREFIX}-k20-dev-small.csv'}

## 지표

In [3]:
gold = load_gold(load_cfg(), 'dev-small')
ranks = {name: gold_ranks(read_run(RUNS / f), gold, 10) for name, f in RUN_FILES.items()}
print(len(gold), 'queries')
table = pd.DataFrame({n: {f'recall@{k}': (r <= k).mean() for k in (1, 5, 10)}
                      for n, r in ranks.items()})
print(table.round(4))

1039 queries
           EXP-011 (k 60)  EXP-014 (k 20)
recall@1           0.9259          0.9259
recall@5           0.9865          0.9875
recall@10          0.9894          0.9904


## 기준 대비 질의별 이동

In [4]:
b, c = ranks['EXP-011 (k 60)'], ranks['EXP-014 (k 20)']
for k in (1, 5):
    gain = int(((b.isna() | (b > k)) & (c <= k)).sum())
    loss = int(((b <= k) & (c.isna() | (c > k))).sum())
    print(f'top{k}: 진입 {gain}, 이탈 {loss}, 순증 {gain - loss}')
print('정답 순위가 바뀐 질의', int((b.fillna(99) != c.fillna(99)).sum()))

top1: 진입 0, 이탈 0, 순증 0
top5: 진입 1, 이탈 0, 순증 1
정답 순위가 바뀐 질의 7
